In [19]:
!pip install langchain faiss-cpu sentence-transformers openai streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 84.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 121.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 113.3 MB/s eta 0:00:00


In [20]:
texts = [
    "Win a free iPhone now",
    "Call me later",
    "Limited offer just for you",
    "Meeting at 5 PM",
    "Congratulations you won lottery"
]

labels = ["spam", "not spam", "spam", "not spam", "spam"]

In [21]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

embeddings = model.encode(texts)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [22]:
import faiss
import numpy as np

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)

index.add(np.array(embeddings))

In [23]:
def retrieve(query):
    q_embed = model.encode([query])
    D, I = index.search(np.array(q_embed), k=2)

    results = [texts[i] for i in I[0]]
    return results

In [24]:
!pip install langchain-openai

In [29]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(openai_api_key="YOUR_API_KEY")

def classify(query):
    context = retrieve(query)
    if any(word in query.lower() for word in ["free", "offer", "win", "money"]):
        return "spam"
    else:
        return "not spam"

    prompt = f"""
    You are a spam classifier.

    Examples:
    {context}

    Classify this message:
    "{query}"

    Answer only: spam or not spam
    """

    response = llm.predict(prompt)
    return response

In [31]:
def retrieve(query):
    q_embed = model.encode([query])

In [30]:
query = "Free money offer just for you"
print(classify(query))

spam


In [33]:
%%writefile app.py
import streamlit as st
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

st.title("Spam Detector (RAG + AI)")

# Data
texts = [
    "Win a free iPhone now",
    "Call me later",
    "Limited offer just for you",
    "Meeting at 5 PM",
    "Congratulations you won lottery"
]

# Load model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Create embeddings
embeddings = model.encode(texts)

# FAISS index
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

# Retrieve
def retrieve(query):
    q_embed = model.encode([query])
    D, I = index.search(np.array(q_embed), k=2)
    return [texts[i] for i in I[0]]

# Classify
def classify(query):
    if any(word in query.lower() for word in ["free", "offer", "win", "money"]):
        return "spam"
    return "not spam"

# UI
user_input = st.text_input("Enter message")

if user_input:
    result = classify(user_input)

    if result == "spam":
        st.error("Spam 🚫")
    else:
        st.success("Not Spam ✅")

Writing app.py


In [35]:
%%writefile requirements.txt
streamlit
sentence-transformers
faiss-cpu
numpy

Overwriting requirements.txt
